# RAG with LangChain and gpt-oss

This notebook builds a Retrieval-Augmented Generation pipeline with LangChain, Chroma, local HuggingFace embeddings, and a Groq-hosted gpt-oss model. It is the hands-on counterpart to the from-scratch notebook, using the LangChain framework.

For the underlying ideas (what RAG is, how it compares to fine-tuning, and where it is used), read [RAG Concepts](1_rag_concepts.md) first.

## Learning Objectives

At the end of this notebook, you should be able to:

- Load and split PDF documents using pypdf and LangChain's text splitter.
- Build a Chroma vector store from HuggingFace embeddings and query it as a retriever.
- Connect a retriever to a Groq-hosted gpt-oss model with LangChain chains to answer questions from the documents.
- Compare the framework workflow with the from-scratch pipeline and identify what LangChain automates.

## Setup

In [1]:
import warnings

from dotenv import load_dotenv
from langchain_groq import ChatGroq

/Users/fritzvonscherzer/Documents/AIPM/aipm-rag-pipeline/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load credentials

In [2]:
load_dotenv()

True

### Defining the LLM (Using Groq)

In [3]:
warnings.filterwarnings("ignore")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

## Implementing RAG with LangChain and gpt-oss

This section builds the RAG system over research papers (an AI paper and a pharmaceutical leaflet). It loads the PDFs, stores their embeddings in a Chroma vector store, retrieves the most relevant passages for a question, and uses a Groq-hosted gpt-oss model to answer from them. The concepts behind each stage are covered in [RAG Concepts](1_rag_concepts.md).

The workflow separates **ingestion**, which prepares and indexes documents, from **inference**, which retrieves context and generates an answer for each question.

![RAG workflow separating ingestion, retrieval, augmentation, and generation](assets/rag_workflow.png)

*RAG pipeline architecture from document ingestion to a grounded response.*

|Technologies|Description|
|---|---|
|[LangChain](https://python.langchain.com/docs/introduction/)|Framework for developing applications powered by LLMs|
|[Chroma](https://www.trychroma.com/)|Open-source vector database for storing contextual embedding vectors and running similarity search|
|[Groq](https://console.groq.com/docs/overview)|Engine providing fast AI inference in the cloud|

### Step 1: Load Data

In [4]:
import shutil
from pathlib import Path

from langchain_chroma import Chroma
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langsmith import Client
from pypdf import PdfReader

In [5]:
def load_pdf_data(pdf_path):
    """
    Load text data from a PDF file, one Document per page.
    """
    reader = PdfReader(pdf_path)
    documents = [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": pdf_path, "page": page_number},
        )
        for page_number, page in enumerate(reader.pages, start=1)
    ]
    return documents

In [6]:
react_docs = load_pdf_data(pdf_path="documents/react_paper.pdf")

In [7]:
# Show number of pages
print(f"number of loaded pages: {len(react_docs)}")

number of loaded pages: 33


In [8]:
# Show page content
print(react_docs[0].page_content)

Published as a conference paper at ICLR 2023
REAC T: S YNERGIZING REASONING AND ACTING IN
LANGUAGE MODELS
Shunyu Yao∗*,1, Jeffrey Zhao2, Dian Yu2, Nan Du2, Izhak Shafran2, Karthik Narasimhan1, Yuan Cao2
1Department of Computer Science, Princeton University
2Google Research, Brain team
1{shunyuy,karthikn}@princeton.edu
2{jeffreyzhao,dianyu,dunan,izhak,yuancao}@google.com
ABSTRACT
While large language models (LLMs) have demonstrated impressive performance
across tasks in language understanding and interactive decision making, their
abilities for reasoning (e.g. chain-of-thought prompting) and acting (e.g. action
plan generation) have primarily been studied as separate topics. In this paper, we
explore the use of LLMs to generate both reasoning traces and task-speciﬁc actions
in an interleaved manner, allowing for greater synergy between the two: reasoning
traces help the model induce, track, and update action plans as well as handle
exceptions, while actions allow it to interface with an

### Step 2: Split Documents into Chunks

In [9]:
def split_documents(documents, chunk_size=200, chunk_overlap=50):
    """
    Splits documents into chunks of given size and overlap
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )
    chunks = text_splitter.split_documents(documents=documents)

    # Just to add id for etch chunks to map it later
    for i, chunk in enumerate(chunks):
        chunk.metadata.update(
            {
                "id": f"chunk_{i}",
            }
        )

    return chunks


react_chunks = split_documents(react_docs)

In [10]:
# Show number of chunks created
print(
    f"number of chunks created: {len(react_chunks)}",
    "\n",
    f"Type of the chunks : {type(react_chunks)}",
    "\n\n",
    react_chunks,
)

number of chunks created: 705 
 Type of the chunks : <class 'list'> 

 [Document(metadata={'source': 'documents/react_paper.pdf', 'page': 1, 'id': 'chunk_0'}, page_content='Published as a conference paper at ICLR 2023\nREAC T: S YNERGIZING REASONING AND ACTING IN\nLANGUAGE MODELS'), Document(metadata={'source': 'documents/react_paper.pdf', 'page': 1, 'id': 'chunk_1'}, page_content='LANGUAGE MODELS\nShunyu Yao∗*,1, Jeffrey Zhao2, Dian Yu2, Nan Du2, Izhak Shafran2, Karthik Narasimhan1, Yuan Cao2\n1Department of Computer Science, Princeton University\n2Google Research, Brain team'), Document(metadata={'source': 'documents/react_paper.pdf', 'page': 1, 'id': 'chunk_2'}, page_content='2Google Research, Brain team\n1{shunyuy,karthikn}@princeton.edu\n2{jeffreyzhao,dianyu,dunan,izhak,yuancao}@google.com\nABSTRACT'), Document(metadata={'source': 'documents/react_paper.pdf', 'page': 1, 'id': 'chunk_3'}, page_content='ABSTRACT\nWhile large language models (LLMs) have demonstrated impressive perfor

The 33-page paper splits into 705 chunks with `chunk_size=200` and `chunk_overlap=50`. Smaller chunks make each retrieval more focused, at the cost of storing and searching more vectors. Larger chunks would give fewer, broader passages.

### Step 3: Generate and Store Embeddings in a VectorStore

In [11]:
def create_embedding_vector_db(chunks, db_name):
    """
    This function uses the open-source embedding model HuggingFaceEmbeddings
    to create embeddings and store those in a Chroma vector store,
    which allows for efficient similarity search
    """
    # instantiate embedding model
    embedding = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2",
        encode_kwargs={"normalize_embeddings": True},
    )
    # start from a clean directory so re-running the cell does not duplicate vectors
    persist_directory = f"vector_databases/vector_db_{db_name}"
    if Path(persist_directory).exists():
        shutil.rmtree(persist_directory)
    # create the vector store, persisted to disk and using cosine distance
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space": "cosine"},  # or "l2" or "ip"
    )
    return vectorstore


all_embedding = create_embedding_vector_db(chunks=react_chunks, db_name="react")

> **Note: why normalise embeddings and set the cosine space in Chroma?**
>
> Cosine similarity compares the direction of two vectors, not their magnitude. If the embeddings are not normalised, the score can be skewed by vector length, which leads to less relevant results.
>
> Chroma defaults to the L2 (Euclidean) distance. Passing `collection_metadata={"hnsw:space": "cosine"}` switches the collection to cosine distance, and setting `normalize_embeddings=True` keeps every vector at unit length so the search reflects direction alone. Using both together makes the retrieval scores reflect true cosine similarity.

### Step 4: Load embedded chunks from VectorStore and make retrieve object

**Why reload the vector store?**

Use the same embedding model that created the stored vectors, so the query is embedded into the same space. The retrieval chain then expects a retriever object as its input, which the vector store provides through `as_retriever`.

In [12]:
def retrieve_from_vector_db(vector_db_path):
    """
    this function splits out a retriever object from a local Chroma vector store
    """
    # instantiate embedding model
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-mpnet-base-v2",
        encode_kwargs={"normalize_embeddings": True},
    )
    react_vectorstore = Chroma(
        persist_directory=vector_db_path,
        embedding_function=embeddings,
    )
    retriever = react_vectorstore.as_retriever()
    return retriever, react_vectorstore


# Load the retriever and index
react_retriever, react_vectorstore = retrieve_from_vector_db(
    "vector_databases/vector_db_react"
)
type(react_retriever), type(react_vectorstore)

(langchain_core.vectorstores.base.VectorStoreRetriever,
 langchain_chroma.vectorstores.Chroma)

In [13]:
query = """
what is react ?
"""

In [14]:
react_retriever.invoke(query)

[Document(id='9116ba1f-b7b8-4a51-b503-c0824890cbbb', metadata={'source': 'documents/react_paper.pdf', 'page': 14, 'id': 'chunk_351'}, page_content='ReAct’s reasoning traces. Figure 5 shows that by simply removing a hallucinating sentence in Act\n17 and adding some hints in Act 23, ReAct can be made to change its behavior drastically to align'),
 Document(id='6e84be35-fa51-4fa4-bd4d-05a5a5a78411', metadata={'id': 'chunk_98', 'source': 'documents/react_paper.pdf', 'page': 4}, page_content='when ﬁnetuning is enabled, and in Section 4 how ReAct performance is robust to prompt selections.\nD) Human aligned and controllable: ReAct promises an interpretable sequential decision making'),
 Document(id='5682ea23-a8f4-4dcb-babe-73a9f3750ad3', metadata={'page': 31, 'source': 'documents/react_paper.pdf', 'id': 'chunk_652'}, page_content='ReAct uses reasoning to ﬁnd products that satisfy all target attributes.\nInstruction: get me a sixteen pack of apple cinnamon freeze dried banana chips, and price

The retriever returns the chunks whose embeddings are closest to the query, which are the passages the next step passes to the model as context.

### Step 5: Connecting the Retriever to LLM and Generate Response

The two chains compose like this: `create_retrieval_chain` wraps the retriever and a `stuff_documents_chain`, which stuffs the retrieved chunks plus the question into a prompt for the LLM.

```mermaid
flowchart LR
    Q[User question] --> RC[retrieval_chain]
    RC --> R[retriever]
    R --> C[Relevant chunks]
    C --> SDC[stuff_documents_chain]
    Q --> SDC
    SDC --> LLM[Groq gpt-oss]
    LLM --> A[Answer]
```

In [15]:
def connect_chains(retriever):
    """
    this function connects stuff_documents_chain with retrieval_chain
    """
    stuff_documents_chain = create_stuff_documents_chain(
        llm=llm,
        prompt=Client().pull_prompt(
            "langchain-ai/retrieval-qa-chat",
            include_model=False,
            dangerously_pull_public_prompt=True,
        ),
    )
    retrieval_chain = create_retrieval_chain(
        retriever=retriever, combine_docs_chain=stuff_documents_chain
    )
    return retrieval_chain


react_retrieval_chain = connect_chains(react_retriever)

Now invoke the chain with a real question about the paper. The result is a dictionary with the `input`, the retrieved `context`, and the `answer`.

In [16]:
# Ask a real question about the paper
output = react_retrieval_chain.invoke({"input": "What is the ReAct framework?"})
print(output["answer"])

**ReAct** is a framework that lets a large language model (LLM) *reason* and *act* in a single, tightly‑coupled loop.  
In a ReAct prompt the model writes out a chain of thoughts (its reasoning) and, at any point, can issue an “action” such as calling an API, looking up a fact, or performing a calculation. The result of that action is then fed back into the next step of reasoning.  

Key points from the context:

| Feature | What it means |
|---------|---------------|
| **Intuitive & easy to design** | Human annotators simply type their thoughts in natural language on top of the actions they want the model to take. No complex scripting is required. |
| **Grounded, fact‑driven behavior** | By interleaving reasoning with real‑world look‑ups (e.g., a Wikipedia API call), the model’s answers are anchored in verifiable data, reducing hallucinations. |
| **Verification loop** | The model can check its own outputs against retrieved facts, leading to higher success rates (e.g., 14 % vs. 6 % in

The answer is generated only from the retrieved chunks, so it reflects what the paper says about ReAct rather than the model's general knowledge.

In [17]:
# Now ask about something the paper never mentions
output = react_retrieval_chain.invoke({"input": "what is a zebra?"})
print(output["answer"])

I’m sorry, but I don’t have enough information in the provided context to answer that.


This second question asks about "zebra", a term the paper never uses, and the chain answers that there is no mention of it in the context rather than inventing one. This is the point of RAG: the model answers only from the retrieved context, so an out-of-scope question yields an honest "not found" instead of a hallucination.

### Step 6: Your Turn

You have built the pipeline on the React paper. Now apply the **same functions** to a different document, the paracetamol leaflet at `documents/paracetamol.pdf`.

Everything you need is already defined: `load_pdf_data`, `split_documents`, `create_embedding_vector_db`, `retrieve_from_vector_db`, and `connect_chains`. You only need to call them in order, then ask the chain a question. The model answers from whatever document you give it, so the pipeline should work unchanged.

#### 1. Load and split the document

In [18]:
paracetamol_docs = load_pdf_data(pdf_path="documents/paracetamol.pdf")
paracetamol_chunks = split_documents(paracetamol_docs)
print(f"pages: {len(paracetamol_docs)}, chunks: {len(paracetamol_chunks)}")

pages: 3, chunks: 130


#### 2. Create and load the vector store

In [19]:
paracetamol_embedding = create_embedding_vector_db(
    chunks=paracetamol_chunks, db_name="paracetamol"
)
paracetamol_retriever, paracetamol_vectorstore = retrieve_from_vector_db(
    "vector_databases/vector_db_paracetamol"
)

#### 3. Connect the chain and ask questions

In [20]:
paracetamol_chain = connect_chains(paracetamol_retriever)

output = paracetamol_chain.invoke(
    {"input": "What is the maximum daily dose of paracetamol for adults?"}
)
print(output["answer"])

# Same question in another language (German) to compare the answers
output_de = paracetamol_chain.invoke(
    {"input": "Was ist die maximale Tagesdosis von Paracetamol für Erwachsene?"}
)
print(output_de["answer"])

I’m sorry, but the provided context does not contain information about the maximum daily dose of paracetamol for adults.
Die maximale Tagesdosis von Paracetamol für Erwachsene beträgt laut den Angaben im Kontext **bis zu 60 mg pro Kilogramm Körpergewicht** (also maximal 60 mg × KG pro Tag).


### Step 7: Semantic search and embedding function

Besides running the full chain, you can query the vector store directly. This is useful for inspecting what retrieval returns and for seeing the similarity scores behind it. These searches reuse `all_embedding`, the React paper store built in Step 3.

In [21]:
# A clear, on-topic query to search the stored React paper vectors for
query = "How does ReAct combine reasoning and acting?"

In [22]:
# Return the chunks most similar to the query
all_embedding.similarity_search(query, k=3)

[Document(id='0e6e763a-ba57-4cd2-ab26-25273af12ca7', metadata={'id': 'chunk_211', 'page': 8, 'source': 'documents/react_paper.pdf'}, page_content='state and what needs to be completed by the agent for the goal to be satisﬁed. In contrast, the\nreasoning traces in ReAct for decision making is ﬂexible and sparse, allowing diverse reasoning'),
 Document(id='4de5ee0d-c4ee-4346-b403-bf947b8262a4', metadata={'page': 6, 'source': 'documents/react_paper.pdf', 'id': 'chunk_150'}, page_content='access of an external knowledge base.\nB) While interleaving reasoning, action and observation steps improves ReAct’s grounded-'),
 Document(id='9116ba1f-b7b8-4a51-b503-c0824890cbbb', metadata={'page': 14, 'source': 'documents/react_paper.pdf', 'id': 'chunk_351'}, page_content='ReAct’s reasoning traces. Figure 5 shows that by simply removing a hallucinating sentence in Act\n17 and adding some hints in Act 23, ReAct can be made to change its behavior drastically to align')]

In [23]:
# The same search, with a distance score per chunk (a lower score means more similar)
all_embedding.similarity_search_with_score(query, k=2)

[(Document(id='0e6e763a-ba57-4cd2-ab26-25273af12ca7', metadata={'page': 8, 'source': 'documents/react_paper.pdf', 'id': 'chunk_211'}, page_content='state and what needs to be completed by the agent for the goal to be satisﬁed. In contrast, the\nreasoning traces in ReAct for decision making is ﬂexible and sparse, allowing diverse reasoning'),
  0.2664620876312256),
 (Document(id='4de5ee0d-c4ee-4346-b403-bf947b8262a4', metadata={'page': 6, 'source': 'documents/react_paper.pdf', 'id': 'chunk_150'}, page_content='access of an external knowledge base.\nB) While interleaving reasoning, action and observation steps improves ReAct’s grounded-'),
  0.27624285221099854)]

In [24]:
# You can also embed the query yourself and search by vector
embedding_query = all_embedding.embeddings.embed_query(query)
all_embedding.similarity_search_by_vector(embedding_query, k=3)

[Document(id='0e6e763a-ba57-4cd2-ab26-25273af12ca7', metadata={'id': 'chunk_211', 'source': 'documents/react_paper.pdf', 'page': 8}, page_content='state and what needs to be completed by the agent for the goal to be satisﬁed. In contrast, the\nreasoning traces in ReAct for decision making is ﬂexible and sparse, allowing diverse reasoning'),
 Document(id='4de5ee0d-c4ee-4346-b403-bf947b8262a4', metadata={'source': 'documents/react_paper.pdf', 'page': 6, 'id': 'chunk_150'}, page_content='access of an external knowledge base.\nB) While interleaving reasoning, action and observation steps improves ReAct’s grounded-'),
 Document(id='9116ba1f-b7b8-4a51-b503-c0824890cbbb', metadata={'page': 14, 'source': 'documents/react_paper.pdf', 'id': 'chunk_351'}, page_content='ReAct’s reasoning traces. Figure 5 shows that by simply removing a hallucinating sentence in Act\n17 and adding some hints in Act 23, ReAct can be made to change its behavior drastically to align')]

## Summary

In this notebook you:

- Loaded and split a PDF with pypdf and LangChain's text splitter.
- Built a Chroma vector store from HuggingFace embeddings and queried it as a retriever.
- Connected the retriever to a Groq-hosted gpt-oss model with LangChain chains to answer questions about the paper.
- Saw RAG stay grounded: an off-topic question returned "not in the context" instead of a hallucination.
- Applied the same pipeline to a new document yourself in the Step 6 exercise.

## References & Further Reading

**Guides and tutorials**

1. [RAG vs. Fine Tuning](https://www.youtube.com/watch?v=00Q0G84kq3M)
2. [How to Use Langchain Chain Invoke: A Step-by-Step Guide](https://medium.com/@asakisakamoto02/how-to-use-langchain-chain-invoke-a-step-by-step-guide-9a6f129d77d1)
3. [Implementing RAG using Langchain and Ollama](https://medium.com/@imabhi1216/implementing-rag-using-langchain-and-ollama-93bdf4a9027c)
4. [A beginner's guide to building a RAG application from scratch](https://learnbybuilding.ai/tutorial/rag-from-scratch/)

**Foundational papers**

5. Lewis, P., et al. (2020). "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks." *NeurIPS 2020*. [arXiv:2005.11401](https://arxiv.org/abs/2005.11401)
6. Gao, Y., et al. (2023). "Retrieval-Augmented Generation for Large Language Models: A Survey." *arXiv preprint*. [arXiv:2312.10997](https://arxiv.org/abs/2312.10997)
7. Ram, O., et al. (2023). "In-Context Retrieval-Augmented Language Models." *ACL 2023*. [arXiv:2302.00083](https://arxiv.org/abs/2302.00083)
8. Borgeaud, S., et al. (2022). "Improving language models by retrieving from trillions of tokens." *ICML 2022*. [arXiv:2112.04426](https://arxiv.org/abs/2112.04426)
9. Izacard, G., & Grave, E. (2021). "Leveraging Passage Retrieval with Generative Models for Open Domain Question Answering." *EACL 2021*. [arXiv:2007.01282](https://arxiv.org/abs/2007.01282)